In [1]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger

from caveat.label_encoding import TokenAttributeEncoder
from caveat.mine_yz import DataModule, MutualInformationEstimator, YZDataset
from caveat.models.continuous.cvae_lstm import LabelEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/torchmetrics/__init__.py:31: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.4)
  import scipy.signal


Device: cuda


/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


In [2]:
def latest(path: Path):
    versions = sorted(
        [
            d
            for d in path.iterdir()
            if d.is_dir() and d.name.startswith("version")
        ]
    )
    return Path(versions[-1])


def iter_models(path: Path):
    for dir in path.iterdir():
        if dir.is_dir() and not dir.name == "eval":
            yield latest(dir)

In [ ]:
label_encoder = TokenAttributeEncoder(
    config={
        "age": "nominal",
        "sex": "nominal",
        "employment": "nominal",
        "hh_income": "nominal",
        "hh_zone": "nominal",
        "day": "nominal",
        "vehicles": "nominal",
        "access_egress_distance": "nominal",
    }
)

ys = pd.read_csv(
    Path("~/Data/foundata/out/nts/2023/attributes_binned.csv"), low_memory=False
)
print(ys.vehicles.value_counts())
label_encoder.build_config(ys)


def custom_loader(
    root: Path, label_encoder, random_z: bool = False, embed_z: bool = False
):
    for path in iter_models(root):
        ys = pd.read_csv(
            path / "test_inference" / "input_attributes.csv", low_memory=False
        )
        # convert vehciles column to strings
        ys.vehicles = ys.vehicles.astype(str)

        ys, _ = label_encoder.encode(ys)
        zs = pd.read_csv(path / "test_inference" / "zs.csv", header=None).values
        if random_z:
            rng = np.random.default_rng()
            zs = rng.normal(loc=0.0, scale=1.0, size=zs.shape)
        if embed_z:
            # strong MI example
            embedder = LabelEncoder(
                label_embed_sizes=label_encoder.label_kwargs[
                    "label_embed_sizes"
                ],
                hidden_size=zs.shape[1],
            )
            zs = embedder(ys).detach().numpy()
        yield ys, zs

Label Encoder TokenAttributeEncoder initialised with:
            Label weighting: unit
            Joint weighting: unit
            
vehicles
1          26715
2          17705
0           8460
3           3119
4            582
5            122
unknown       12
9             11
6              8
Name: count, dtype: int64


In [4]:
class MinerNet(nn.Module):
    def __init__(
        self,
        label_embed_sizes,
        hidden_size=512,
        latent_dim=6,
        block_depth=4,
        dropout=0.3,
    ):
        super(MinerNet, self).__init__()

        self.label_embed = LabelEncoder(
            label_embed_sizes=label_embed_sizes, hidden_size=hidden_size
        )
        self.z_embed = nn.Sequential(
            nn.Linear(in_features=latent_dim, out_features=hidden_size),
            nn.LeakyReLU(),
        )

        blocks = []
        for _ in range(block_depth - 1):
            blocks.append(nn.Linear(hidden_size, hidden_size))
            if dropout > 0:
                blocks.append(nn.Dropout(dropout))
            blocks.append(nn.LeakyReLU())
        self.blocks = nn.Sequential(*blocks, nn.Linear(hidden_size, 1))

    def forward(self, ys, zs):
        h1 = self.label_embed(ys.long())
        h2 = self.z_embed(zs)
        return self.blocks(h1 + h2)

In [5]:
data_loaders = {
    "random": custom_loader(
        Path("../logs/actvae/cpvae"), label_encoder, random_z=True
    ),
    "cpvae": custom_loader(Path("../logs/actvae/cpvae"), label_encoder),
    "vae": custom_loader(Path("../logs/actvae/vae"), label_encoder),
    "strong": custom_loader(
        Path("../logs/actvae/vae"), label_encoder, embed_z=True
    ),
}
results = {}
for name, loader in data_loaders.items():
    model_results = []
    for i, (ys, zs) in enumerate(loader):

        logger = TensorBoardLogger("logs/zy", name=f"{name}_{i}")
        dataset = YZDataset(ys=ys, zs=zs)
        loader = DataModule(
            dataset=dataset,
            val_split=0.1,
            test_split=0.1,
            batch_size=1024,
            num_workers=8,
            pin_memory=False,
        )

        net = MinerNet(
            label_embed_sizes=label_encoder.label_kwargs["label_embed_sizes"],
            hidden_size=512,
            block_depth=6,
            latent_dim=6,
            dropout=0.3,
        )

        kwargs = {"alpha": 1, "lr": 1e-3, "weight_decay": 1e-3}
        model = MutualInformationEstimator(net=net, **kwargs)
        trainer = Trainer(
            min_epochs=10,
            max_epochs=500,
            accelerator=device,
            devices=1,
            enable_progress_bar=False,
            logger=logger,
            enable_checkpointing=True,
            callbacks=[
                EarlyStopping(monitor="val_loss", patience=20),
                ModelCheckpoint(
                    monitor="val_loss", save_top_k=2, save_weights_only=False
                ),
            ],
        )
        trainer.fit(model, datamodule=loader)
        mi = trainer.test(ckpt_path="best", datamodule=loader)[0]["test_mi"]
        model_results.append(mi)
    results[name] = {
        "mean": np.mean(model_results),
        "var": np.var(model_results),
    }

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net         │ MinerNet │  1.3 M │ train │     0 │
│ 1 │ energy_loss │ Mine     │  1.3 M │ train │     0 │
└───┴─────────────┴──────────┴────────┴───────┴───────┘

Trainable params: 1.3 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.3 M                                                                                                
Total estimated model params size (MB): 5                                                                          
Modules in train mode: 32                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:317: The number of training batches (5) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │  -0.0071048736572265625   │
│          test_mi          │   0.0071048736572265625   │
└───────────────────────────┴───────────────────────────┘

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    -6.180408954620361     │
│          test_mi          │     6.180408954620361     │
└───────────────────────────┴───────────────────────────┘

┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net         │ MinerNet │  1.3 M │ train │     0 │
│ 1 │ energy_loss │ Mine     │  1.3 M │ train │     0 │
└───┴─────────────┴──────────┴────────┴───────┴───────┘

Trainable params: 1.3 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.3 M                                                                                                
Total estimated model params size (MB): 5                                                                          
Modules in train mode: 32                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:317: The number of training batches (5) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    -7.687297821044922     │
│          test_mi          │     7.687297821044922     │
└───────────────────────────┴───────────────────────────┘

In [6]:
for name, result in results.items():
    print(f"\tResults for {name}: {result}")

    # 	Results for random: {'mean': 6.536468863487244e-05, 'var': 5.342797632576502e-09}
# Results for cvae: {'mean': 0.17016934156417846, 'var': 0.008924022788872266}
# Results for vae: {'mean': 0.3406977355480194, 'var': 0.0015793720033414616}
# Results for strong: {'mean': 3.9802443027496337, 'var': 0.8028825184150626}

	Results for random: {'mean': np.float64(0.0014179341495037078), 'var': np.float64(8.091376961718577e-06)}
	Results for cpvae: {'mean': np.float64(0.41029676198959353), 'var': np.float64(0.0031405181508714726)}
	Results for vae: {'mean': np.float64(0.3973314881324768), 'var': np.float64(0.0052781562620113935)}
	Results for strong: {'mean': np.float64(6.491370296478271), 'var': np.float64(2.956886334049559)}


In [7]:
df = pd.DataFrame.from_dict(results, orient="index")
print(df.to_latex(float_format="{:.4f}".format))

# \begin{tabular}{lrr}
# \toprule
#  & mean & var \\
# \midrule
# random & 0.0001 & 0.0000 \\
# cvae & 0.1702 & 0.0089 \\
# vae & 0.3407 & 0.0016 \\
# strong & 3.9802 & 0.8029 \\
# \bottomrule
# \end{tabular}

\begin{tabular}{lrr}
\toprule
 & mean & var \\
\midrule
random & 0.0014 & 0.0000 \\
cpvae & 0.4103 & 0.0031 \\
vae & 0.3973 & 0.0053 \\
strong & 6.4914 & 2.9569 \\
\bottomrule
\end{tabular}

